In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
application_test = pd.read_csv("home-credit-default-risk/application_test.csv")
application_train = pd.read_csv("home-credit-default-risk/application_train.csv")
bureau = pd.read_csv("home-credit-default-risk/bureau.csv")
bureau_balance = pd.read_csv("home-credit-default-risk/bureau_balance.csv")
credit_card_balance = pd.read_csv("home-credit-default-risk/credit_card_balance.csv")
installments_payments = pd.read_csv("home-credit-default-risk/installments_payments.csv")
POS_CASH_balance = pd.read_csv("home-credit-default-risk/POS_CASH_balance.csv")
previous_application = pd.read_csv("home-credit-default-risk/previous_application.csv")
sample_submission = pd.read_csv("home-credit-default-risk/sample_submission.csv")

In [3]:
#bureau.csv

In [13]:
bureau.shape

(23495, 15)

In [5]:
bureau.columns

Index(['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY',
       'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE',
       'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG',
       'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT',
       'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE',
       'AMT_ANNUITY'],
      dtype='object')

In [16]:
bureau.head(15)

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0.0,91323.00,0.00,NaN,0.0,Consumer credit,-131.0,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0.0,225000.00,171342.00,NaN,0.0,Credit card,-20.0,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0.0,464323.50,NaN,NaN,0.0,Consumer credit,-16.0,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0.0,90000.00,NaN,NaN,0.0,Credit card,-16.0,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0.0,2700000.00,NaN,NaN,0.0,Consumer credit,-21.0,NaN
5,215354,5714467,Active,currency 1,-273,0,27460.0,NaN,0.0,0.0,180000.00,71017.38,108982.62,0.0,Credit card,-31.0,NaN
6,215354,5714468,Active,currency 1,-43,0,79.0,NaN,0.0,0.0,42103.80,42103.80,0.00,0.0,Consumer credit,-22.0,NaN
7,162297,5714469,Closed,currency 1,-1896,0,-1684.0,-1710.0,14985.0,0.0,76878.45,0.00,0.00,0.0,Consumer credit,-1710.0,NaN
8,162297,5714470,Closed,currency 1,-1146,0,-811.0,-840.0,0.0,0.0,103007.70,0.00,0.00,0.0,Consumer credit,-840.0,NaN
9,162297,5714471,Active,currency 1,-1146,0,-484.0,NaN,0.0,0.0,4500.00,0.00,0.00,0.0,Credit card,-690.0,NaN


In [14]:
bureau["CREDIT_ACTIVE"].value_counts()

CREDIT_ACTIVE
Closed      14397
Active       9014
Sold           83
Bad debt        1
Name: count, dtype: int64

In [12]:
bureau.isnull().sum().sort_values(ascending=False)

DAYS_ENDDATE_FACT         9052
AMT_CREDIT_SUM_LIMIT      8203
AMT_CREDIT_SUM_DEBT       3536
DAYS_CREDIT_ENDDATE       1572
SK_ID_CURR                   0
SK_ID_BUREAU                 0
CREDIT_ACTIVE                0
CREDIT_CURRENCY              0
DAYS_CREDIT                  0
CREDIT_DAY_OVERDUE           0
CNT_CREDIT_PROLONG           0
AMT_CREDIT_SUM               0
AMT_CREDIT_SUM_OVERDUE       0
CREDIT_TYPE                  0
DAYS_CREDIT_UPDATE           0
dtype: int64

In [8]:
#AMT_ANNUITY und AMT_CREDIT_MAX_OVERDUE haben mehr als 50% NA Werte
bureau.drop(columns=['AMT_ANNUITY', "AMT_CREDIT_MAX_OVERDUE"], inplace=True)

In [11]:
#Die Spalten hier haben nur einen einzigen NA Wert. Diese Zeilen droppen wir einfach
bureau = bureau.dropna(subset=["CNT_CREDIT_PROLONG"])
bureau = bureau.dropna(subset=["AMT_CREDIT_SUM"])
bureau = bureau.dropna(subset=["AMT_CREDIT_SUM_OVERDUE"])
bureau = bureau.dropna(subset=["CREDIT_TYPE"])
bureau = bureau.dropna(subset=["DAYS_CREDIT_UPDATE"])

In [10]:
bureau["CREDIT_ACTIVE"].value_counts()

CREDIT_ACTIVE
Closed      14398
Active       9014
Sold           83
Bad debt        1
Name: count, dtype: int64

In [ ]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    bureau_loan_count=("SK_ID_BUREAU", "count"),    #Anzahl Kredite
    bureau_active_count=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()), #Anzahl Aktive Kredite
    bureau_total_credit=("AMT_CREDIT_SUM", "sum"),  #Gesamtkreditehöhe
    bureau_total_debt=("AMT_CREDIT_SUM_DEBT", "sum"),   #Gesamtschulden
    bureau_current_debt=("AMT_CREDIT_SUM_OVERDUE", "sum"),  #Aktuelle Schulden
).reset_index()

application_train = application_train.merge(bureau_agg, on="SK_ID_CURR", how="left")